In [ ]:
from dataclasses import dataclass
from typing import Sequence, Any

import numpy as np
import matplotlib.animation as animation
import matplotlib.pyplot as plt
from numpy.typing import NDArray

In [ ]:
"""Define common types and constants that may be used throughout the notebook"""

type PixCoords = tuple[NDArray[np.int64], NDArray[np.int64]]
type PixOffsets = tuple[int, int]
type PSF = NDArray[np.float64]
type Dirty = NDArray[np.float64]
type Residual = NDArray[np.float64]
type Image = NDArray[np.float64]
type Model = NDArray[np.float64]

SAMPLING = 5

@dataclass
class Source:
    x: int | float
    """Location in pixel grid in first dimension"""
    y: int | float
    """Location in pixel grid in second dimension"""
    flux: float
    """Brightness of the peak response"""

type Sources = Sequence[Source]




In [ ]:
def plot_coords(pix_coords: PixCoords) -> None:
    """Sanity check the coordinate system"""

    print(f"The dimension of PixCoords: {len(pix_coords)}")
    for idx, direction in enumerate(pix_coords):
        print(f"{idx} - {direction.shape}")
    
    fig, axes = plt.subplots(1,2)
        
    axes[0].imshow(pix_coords[0], cmap="bwr")
    axes[0].set(title="x-coordinates")
    cim = axes[1].imshow(pix_coords[1], cmap="bwr")
    axes[1].set(title="y-coordinates")
    
    fig.colorbar(cim, label="Coordinate Value")
    fig.tight_layout()

    
    
def make_coord_grid(nx: int, ny: int) -> PixCoords:
    """Make up an initial pixel coordinate grid with the origin at the center

    Args:
        nx (int): Size of the image in the x-direction
        ny (int): Size of the image in the y-direction

    Returns:
        PixCoords: The coordinate grid
    """
    pix_x = np.arange(nx) - nx // 2
    pix_y = np.arange(ny) - ny // 2
    
    coords = np.meshgrid(pix_x.astype(int), pix_y.astype(int))

    return coords 


def plot_psf(psf: PSF, cbar: bool = True) -> None:
    """Sanity check the PSF"""

    fig, ax = plt.subplots(1,1)
    
    cim = ax.imshow(psf, cmap="bwr")
    if cbar:
        fig.colorbar(cim, label="Intensity")



def radial_sinc(pix_coords: PixCoords, pix_offsets: PixOffsets | None = None) -> PSF:
    """Make up a PSF of the system"""

    if pix_offsets:
        # This will shift the origin of the response
        pix_coords = (
            pix_coords[0] - pix_offsets[0],
            pix_coords[1] - pix_offsets[1],
        )

    radius = np.sqrt(pix_coords[0]**2 + pix_coords[1]**2)
    
    radius /= SAMPLING
    
    return np.sinc(radius)


pix_coords = make_coord_grid(nx=100, ny=100)
plot_coords(pix_coords=pix_coords)

psf = radial_sinc(pix_coords=pix_coords)
plot_psf(psf=psf)
    




In [ ]:
def plot_sinc_offsets(pix_coords: PixCoords, offsets: list[int] | None = None) -> None:
    """A simple helper function to display our the PSF function
    can be shifted around"""

    if offsets is None:
        offsets = [-40, 0, 40]
    
    n_offsets = len(offsets)
    fig, axes = plt.subplots(n_offsets, n_offsets)
    
    for i, x_offset in enumerate(offsets):
        for j, y_offset in enumerate(offsets):
            ax = axes[i, j]
            pix_offset = (x_offset, y_offset)
            print(f"Plotting shifted response {pix_offset}")
            response = radial_sinc(
                pix_coords=pix_coords, pix_offsets=pix_offset
            )

            ax.imshow(response, cmap="bwr")


plot_sinc_offsets(pix_coords=pix_coords)




In [ ]:
def make_source_response(pix_coords: PixCoords, source: Source) -> Image:
    """Helper to generate the source response for a given pixel grid and source specificaiton"""
    source_offset = (source.x, source.y)
    source_response = radial_sinc(
        pix_coords=pix_coords, pix_offsets=source_offset
    )

    return source_response * source.flux
    

def get_sources() -> Sources:
    sources = [
        Source(x=-20, y=12, flux=1.0),
        Source(x=32, y=29, flux=0.3),
        Source(x=-8, y=-20, flux=0.7),
    ]
    print(f"Have created {len(sources)} sources")
    return sources

def plot_dirty(dirty: Dirty) -> None:
    """Make a quick look plot of the dirty image"""


    fig, ax = plt.subplots(1, 1)
    
    cim = ax.imshow(dirty, cmap="bwr")
    fig.colorbar(cim, label="Intensity")
    

def make_source_image(pix_coords: PixCoords, sources: Sources) -> Dirty:
    """Generate the image of sources in the field"""
    dirty = np.zeros(pix_coords[0].shape, dtype=np.float64)
    
    for idx, source in enumerate(sources):
        print(f"Generating source {idx}")    
        source_response = make_source_response(
            pix_coords=pix_coords, source=source
        )
        dirty += source_response
        
    return dirty

        
sources = get_sources()
dirty = make_source_image(pix_coords=pix_coords, sources=sources)
plot_dirty(dirty=dirty_image)    



In [ ]:
@dataclass
class CleanOptions:
    max_iterations: int
    """The maximum number of cleaning cycles that are allowed to be performed"""
    stopping_flux: float
    """The stopping flux level"""
    loop_gain: float
    """The gain factor to use per loop, e.g. residual = peak * loop_gain * PSF"""
    plot_iterations: bool = False
    """Plot throughout the cleaning"""

@dataclass
class CleanResults:
    """Simple container to hold outputs from the cleaning process"""
    residual: Residual
    """The cleaning residuals"""
    model: Model
    """The clean components"""
    dirty: Dirty
    """The initial image that needed to be cleaned"""
    frames: Any | None = None
    """Matplotlib axes frames to plot"""


@dataclass
class PeakInfo:
    """General information around the peak found"""
    location: tuple[int, int]
    """"The locaiton fo the peak"""
    offsets: PixOffsets
    """The offset location of the source relative to the reference pixel possiont"""
    flux: float
    """How bright the peak was"""
    iteration: int
    """The iteration count"""

def find_peak(residual: Residual, iteration: int) -> PeakInfo:
    """Find the locaiton of the peak asbolute in the residual   """

    peak_location = np.argmax(np.abs(residual))
    peak_location = np.unravel_index(peak_location, residual.shape)
    peak_flux = residual[peak_location]
    
    # Now calculate the offsets relative to the center of the iamge
    pix_offset = (
        peak_location[0] - residual.shape[0] // 2,
        peak_location[1] - residual.shape[1] // 2,
    )
    
    peak_info = PeakInfo(
        location=peak_location,
        offsets=pix_offset[::-1],
        flux=peak_flux,
        iteration=iteration
    )
    print(f"Iteration {iteration}, on peak flux {peak_flux=:.3f} at offset {pix_offset}")
    
    return peak_info
    
def shift_scale_psf_minor_loop(
    pix_coords: PixCoords, peak_info: PeakInfo, loop_gain: float
) -> Image:
    """Make a shifted and scaled version of the PSF appropriate to subtract
    with in each minor loop"""

    scaled_psf = radial_sinc(
        pix_coords=pix_coords, 
        pix_offsets=peak_info.offsets
    )
    scaled_psf *= (peak_info.flux * loop_gain)
    return scaled_psf
    
    
    
def plot_iteration(
    axes: tuple[plt.Axes,plt.Axes], residual: Residual, response: NDArray[np.float64], peak_info: PeakInfo, v_max: float | None = None
) -> None:
    
    # fig, (ax1, ax2) = plt.subplots(1,2)
    
    ax1, ax2 = axes
    
    ax1.imshow(residual, vmax=v_max)
    ax1.set(title="Residual")
    ax2.imshow(response)
    ax2.set(title="Scaled PSF")

    # fig.suptitle(f"Iteration: {peak_info.iteration}, Peak: {peak_info.flux:.4f}")
    
    return (ax1, ax2)
    
    
    
def clean(pix_coords: PixCoords, dirty: Dirty, clean_options: CleanOptions) -> Model:
    
    frames = []
    if clean_options.plot_iterations:
        plt.close("all")
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    
    
    print("Initialising the residual image")
    residual = dirty.copy()
    model = np.zeros_like(residual)
    
    v_max = np.max(residual)
    
    iteration = 0
    while True:
        
        
        iteration += 1
        
        peak_info = find_peak(residual=residual, iteration=iteration)
        
        scaled_psf = shift_scale_psf_minor_loop(
            pix_coords=pix_coords, peak_info=peak_info, loop_gain=clean_options.loop_gain
        )
        model[peak_info.location] += peak_info.flux
        
        if clean_options.plot_iterations:
            _axes = plot_iteration(
                axes=axes, residual=residual, response=scaled_psf, peak_info=peak_info, v_max=v_max
            )
            frames.append(_axes)
        
        
        residual -= scaled_psf
        
        
        if iteration >= clean_options.max_iterations:
            print("Maximum number of iterations reached")
            break
        if peak_info.flux < clean_options.stopping_flux:
            print("Have reached the clean threshold")
            break

    return CleanResults(
        residual=residual, model=model, dirty=dirty, frames=(fig, frames)
    )

    
clean_options = CleanOptions(
    max_iterations=40, 
    stopping_flux=0.001, 
    loop_gain=0.1,
    plot_iterations=True,
)

clean_results = clean(pix_coords=pix_coords, dirty=dirty, clean_options=clean_options)

if clean_results.frames is not None:
    clean_animation = animation.ArtistAnimation(
        *clean_results.frames, blit=True, interval=50
    )
    clean_animation.save("clean_animation.mp4", writer="ffmpeg", fps=30)
    